In [ ]:
import tensorflow as tf
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2_as_graph

In [ ]:
# Define your LSTM model
def lstm_model(input_shape, lstm_units=32, learning_rate=0.01):
    inputs = Input(shape=input_shape, name="input_layer")
    lstm_out = LSTM(lstm_units, return_sequences=False, name="lstm_layer")(inputs)
    outputs = Dense(1, activation="sigmoid", name="output_layer")(lstm_out)

    model = Model(inputs=inputs, outputs=outputs)
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss="binary_crossentropy", metrics=["accuracy"])
    return model

In [ ]:
# FLOP calculation function
def get_flops(model):
    if isinstance(model, (tf.keras.models.Model, tf.keras.models.Sequential)):
        run_meta = tf.compat.v1.RunMetadata()
        opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()

        # Create concrete function
        inputs = [tf.TensorSpec([1] + list(inp.shape[1:]), inp.dtype) for inp in model.inputs]
        real_model = tf.function(model).get_concrete_function(inputs)

        # Convert variables to constants
        frozen_func, _ = convert_variables_to_constants_v2_as_graph(real_model)

        # Calculate FLOPs with TF 1.x profiler
        flops = tf.compat.v1.profiler.profile(
            graph=frozen_func.graph,
            run_meta=run_meta,
            options=opts,
            cmd='op' # scope
        )

        return flops.total_float_ops if flops is not None else 0
    else:
        raise ValueError("Model must be a tf.keras.Model or tf.keras.Sequential instance.")

In [ ]:
# Instantiate model and calculate FLOPs
input_shape = (3, 14)
model = lstm_model(input_shape)
model.summary()
flops = get_flops(model)
print(f"Estimated FLOPs: {flops}")

In [ ]:
from tensorflow.python.profiler.model_analyzer import profile
from tensorflow.python.profiler.option_builder import ProfileOptionBuilder


forward_pass = tf.function(
    model.call,
    input_signature=[tf.TensorSpec(shape=(1,) + model.input_shape[1:])])

graph_info = profile(forward_pass.get_concrete_function().graph,
                        options=ProfileOptionBuilder.float_operation())

flops = graph_info.total_float_ops 
print('Flops: {:,}'.format(flops))
     